In [6]:
import pandas as pd

df = pd.read_excel('../data/Dataset/online_retail_II.xlsx')

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [7]:
df.shape

(525461, 8)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


In [9]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

In [10]:
df = df.dropna(subset=['Customer ID'])

In [11]:
df.isnull().sum()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64

In [12]:
df = df[~df['Invoice'].astype(str).str.startswith('C')]

In [13]:
df.shape

(407695, 8)

In [14]:
df['TotalPrice'] = df['Quantity'] * df['Price']

In [15]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [31]:
customer_df = df.groupby('Customer ID').agg({
    'Invoice': 'nunique',
    'Quantity': 'sum',
    'TotalPrice': 'sum'
}).reset_index()

In [32]:
customer_df.head()

,Customer ID,Invoice,Quantity,TotalPrice
0,12346.0,11,70,372.86
1,12347.0,2,828,1323.32
2,12348.0,1,373,222.16
3,12349.0,3,993,2671.14
4,12351.0,1,261,300.93


In [33]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled_data = scaler.fit_transform(customer_df[['Invoice', 'Quantity', 'TotalPrice']])

In [34]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, random_state=42)
customer_df['Cluster'] = kmeans.fit_predict(scaled_data)

In [35]:
customer_df.head()

,Customer ID,Invoice,Quantity,TotalPrice,Cluster
0,12346.0,11,70,372.86,0
1,12347.0,2,828,1323.32,0
2,12348.0,1,373,222.16,0
3,12349.0,3,993,2671.14,0
4,12351.0,1,261,300.93,0


In [36]:
customer_df['Cluster'].value_counts()

Cluster
0    3980
3     306
1      22
2       4
4       2
Name: count, dtype: int64

In [38]:
customer_df.groupby('Cluster').mean()

,Customer ID,Invoice,Quantity,TotalPrice
Cluster,,,,
0,15351.632663,3.002010,609.803266,1062.340781
1,15374.727273,61.590909,38619.909091,43973.540909
2,14165.750000,101.500000,131080.500000,128563.190000
3,15319.990196,17.447712,4717.937908,8250.508026
4,16374.000000,83.500000,147279.000000,298780.425000


In [39]:
customer_df.to_csv('customers_clustered.csv', index=False)

In [40]:
X = customer_df[['Invoice', 'Quantity', 'TotalPrice']]
y = customer_df['Cluster']

In [41]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [42]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier()

model.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [43]:
y_pred = model.predict(X_test)

In [44]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print(accuracy)

0.9884125144843569
